In [ ]:
%%capture
%pip install -U ollama langchain_community langchain_huggingface sentence-transformers transformers faiss-cpu

In [ ]:
import ollama
import pathlib
import os
import subprocess
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
def load_context():
    with open('../prompts/llm_context.txt', 'r', encoding='utf-8') as f:
        return f.read()

def update_context(question, answer):
    with open('../prompts/llm_context.txt', 'a', encoding='utf-8') as f:
        f.write("\n\n[Derived Knowledge]\n")
        f.write(f"Question: {question}\n")
        f.write(f"Answer: {answer}\n")
        
def initial_summary(answer):
    with open('../content/prompts/llm_context.txt', 'a', encoding='utf-8') as f:
        f.write("\n\n[Summary]\n")
        f.write(f"{answer}\n")

In [ ]:
system_prompt_tool = """
## Role
You are a Senior Race Engineer and Forza Horizon 5 Expert. Your goal is to synthesize telemetry data and segment-wise feedback into a cohesive, professional race review.

## Task
1. Analyze the provided telemetry segments and the initial 'Response' notes.
2. Identify the most critical recurring mistakes (e.g., poor corner entry speed, over-braking).
3. **TOOL USE:** For any gap involving speed, braking or position call `query_forza_community_knowledge` to obtain verifiable expert tips, tuning setups or community tips for that car/class to help the user fix the issue.
4. Provide the top 5 gap summary that is verifiable based on the telemetry and the retrieved knowledge.

## Style & Constraints (Strict)
- Use English, present tense, and active verbs.
- Prohibited: First-person ("I", "Me") and "Man" form.
- No emojis. No conversational filler.
- Be concise, precise, and professional. Use strong verbs.
- Cite sources (e.g., [Source: Forum]) for all tuning and driving advice.
- Do not disparage other scientific or technical works.
- Focus on "why" the time was lost and "how" to fix it using tuning or technique.
- FINAL OUTPUT: Exactly one structured paragraph detailing the top 5 improvements.
"""

system_prompt = """
## Role
You are a Senior Race Engineer and Forza Horizon 5 Expert. You answer follow-up questions based on a previously generated race summary and telemetry data.

## Task
1. Answer user questions concisely based on the provided context.
2. **TOOL USE:** If the user asks "how" to improve specific car behavior or asks for tuning fixes (e.g., "How to fix oversteer?") or how to get faster in general or corners, you MUST call `query_forza_expert_knowledge`.
3. Synthesize the expert knowledge into a brief, actionable instruction.

## Style & Constraints (Strict)
- **Max Length:** 2 to 3 sentences per response. 
- **Voice:** Professional, direct, no filler, no emojis.
- **Language:** English, present tense, no first-person ("I", "me") or "man".
- **Veracity:** Only make verifiable statements based on telemetry or tool results. Cite sources [e.g., Forum].
- **No Summaries:** Do not repeat the top 5 improvements unless explicitly asked. Focus only on the current question.
"""

model = "nemotron-3-nano:30b"

In [ ]:
# Setting up RAG + tool function
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    folder_path="../content/rag_db/",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

def query_forza_expert_knowledge(question: str) -> str:
    """
    Accesses a specialized database of Forza Horizon forum discussion, tuning guides
    and tips how to get faster. Use this tool specifically for technical tuning details,
    for questions regarding driving faster or what cars are specifically good.
    Not only for expert questions, but also general questions.

    Args:
        query: The specific search query,
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    retrieved_docs = retriever.invoke(question)

    rag_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return rag_context

In [ ]:
llm_context = load_context()

messages = [
            {"role": "assistant", "content": f"Context: {llm_context}" },
            {"role": "system", "content": system_prompt}
        ]

resp = ollama.chat(
    model=model,
    messages=messages,
    think=True
)

answer = resp["message"]["content"]

initial_summary(answer)

In [ ]:
while True:
    follow_up = input("Rückfrage: ")

    if follow_up.lower() == "exit":
        break

    messages = [
        {"role": "assistant", "content": f"Context:\n{llm_context}"},
        {"role": "system", "content": system_prompt_tool},
        {"role": "user", "content": f"{follow_up}"}
    ]
    print("1. Call executed")
    resp = ollama.chat(
        model=model,
        messages=messages,
        tools=[query_forza_expert_knowledge],
        think=True
    )
    print("1. Call finished")

    if resp.message.tool_calls:
        print("2. Call (LLM has used tool)")
        messages.append(resp.message)
        call = resp.message.tool_calls[0]
        query = call.function.arguments['question']
        result = query_forza_expert_knowledge(query)
        
        messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

        final_response = ollama.chat(
            model=model,
            messages=messages,
            tools=[query_forza_expert_knowledge],
            think=True
        )

        answer = final_response["message"]["content"]
        print(final_response["message"])
    else:
        print("Tool call has not been executed")
        answer = resp["message"]["content"]

    print(answer)

    update_context(follow_up, answer)